<a href="https://colab.research.google.com/github/Rahul9994/ML_Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
REPO_URL = "https://github.com/Rahul9994/ML_Flyrank"
REPO_DIR = "ML_Flyrank"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check repo path"
print("Repo cloned and CSV found. Ready to go.")

Cloning into 'ML_Flyrank'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 90 (delta 12), reused 77 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 1.83 MiB | 13.71 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Working dir: /content/ML_Flyrank
Repo cloned and CSV found. Ready to go.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [4]:
'''My lane (Ranking Signal Analysis) is fundamentally a scoring problem. I want to predict
a continuous outcome — engagement (CTR) — from observable content and search signals
(position_tier, word_count, content_type, search_volume, etc.), then use the model's
feature importances to answer the real question: which signals actually matter. This is
scoring rather than classification because CTR is a continuous rate, not a category, and
it's not ranking or clustering because I'm not ordering a list of items against each
other or grouping pages into unlabeled segments — I'm predicting a value per page.'''

"My lane (Ranking Signal Analysis) is fundamentally a scoring problem. I want to predict \na continuous outcome — engagement (CTR) — from observable content and search signals \n(position_tier, word_count, content_type, search_volume, etc.), then use the model's \nfeature importances to answer the real question: which signals actually matter. This is \nscoring rather than classification because CTR is a continuous rate, not a category, and \nit's not ranking or clustering because I'm not ordering a list of items against each \nother or grouping pages into unlabeled segments — I'm predicting a value per page."

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [5]:
'''Target: ctr (click-through rate), an observed outcome already computed in the dataset
from clicks_90d / impressions_90d — not a proxy I'm defining myself, it's a real
measured behavior.

I considered engagement_rate as an alternative target, but ctr is more directly tied to
the action a content editor can take (improving snippets/titles to earn more clicks at
a given position), so it's the cleaner choice for this lane.'''

"Target: ctr (click-through rate), an observed outcome already computed in the dataset \nfrom clicks_90d / impressions_90d — not a proxy I'm defining myself, it's a real \nmeasured behavior.\n\nI considered engagement_rate as an alternative target, but ctr is more directly tied to \nthe action a content editor can take (improving snippets/titles to earn more clicks at \na given position), so it's the cleaner choice for this lane."

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [7]:
'''Success metric: R² (coefficient of determination) on held-out data, alongside MAE (mean
absolute error in CTR units) so the error is interpretable in real terms, not just an
abstract score.

Good would mean: R² meaningfully above 0 (the model explains real variance beyond just
guessing the average CTR), and MAE small enough that a predicted CTR is close enough to
trust for prioritization — not for precise forecasting.'''

'Success metric: R² (coefficient of determination) on held-out data, alongside MAE (mean \nabsolute error in CTR units) so the error is interpretable in real terms, not just an \nabstract score.\n\nGood would mean: R² meaningfully above 0 (the model explains real variance beyond just \nguessing the average CTR), and MAE small enough that a predicted CTR is close enough to \ntrust for prioritization — not for precise forecasting.'

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page
lane_slice = df[[
    "content_id", "position_tier", "word_count", "content_type",
    "search_volume", "avg_position", "ctr"
]].copy()

print("Unit of analysis: one row = one content page")
print(f"Rows: {len(lane_slice)}")
lane_slice.head(10)

Unit of analysis: one row = one content page
Rows: 30000


,content_id,position_tier,word_count,content_type,search_volume,avg_position,ctr
0,content_304f48230142,striking,3221.0,keyword article,10.0,10.6,0.76
1,content_a1fb4e703a9e,page_3_5,2481.0,keyword article,90.0,20.3,0.05
2,content_9aa793d4d895,page_3_5,3515.0,keyword article,0.0,36.5,0.09
3,content_331d6c4de07b,page_1,NaN,keyword article,10.0,6.2,0.49
4,content_d99b7a2d90ca,page_3_5,2803.0,keyword article,0.0,44.0,0.13
5,content_d4084a4bc775,page_1,3080.0,keyword article,720.0,8.5,0.03
6,content_9a34b442b552,page_1,3059.0,keyword article,0.0,7.0,0.00
7,content_a63219c6e95a,page_3_5,NaN,keyword article,590.0,21.2,0.06
8,content_5e6c160719bc,page_3_5,3807.0,keyword article,0.0,46.0,0.09
9,content_c27558df2b0c,page_1,NaN,keyword article,0.0,4.9,0.16


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [10]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].agg(["mean", "std"])
print(ctr_by_tier.round(3))

'''A fixed if-statement (e.g. "if position_tier == 'page_1', predict high CTR") ignores
real spread: the std shown above proves CTR varies a lot even within the same position
tier, so position alone isn't a clean rule. Real CTR depends on an interacting mix of
position, content_type, word_count, and search intent together — that's exactly the
kind of multi-signal, non-additive pattern a fixed rule can't capture but a model can
learn from data.'''

                mean    std
position_tier              
deep           0.055  0.170
page_1         0.355  0.502
page_3_5       0.142  0.229
striking       0.256  0.347
top_3          0.334  0.487


'A fixed if-statement (e.g. "if position_tier == \'page_1\', predict high CTR") ignores \nreal spread: the std shown above proves CTR varies a lot even within the same position \ntier, so position alone isn\'t a clean rule. Real CTR depends on an interacting mix of \nposition, content_type, word_count, and search intent together — that\'s exactly the \nkind of multi-signal, non-additive pattern a fixed rule can\'t capture but a model can \nlearn from data.'

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.